# Modul 20: Fairer Modellvergleich und verantwortungsvolles Abschlussprojekt | Lösungen

## Überblick

Sie planen ein vollständiges ML-Projekt, validieren Daten, vergleichen Baseline und Modelle unter identischen Bedingungen und wählen Schwellen ausschließlich mit Validierungsdaten. Danach prüfen Sie Ressourcen, Teilgruppen, Drift, sichere Inferenz, Persistenz und dokumentieren das Ergebnis in einer kompakten Modellkarte.

**Zugehörige Vorlesungen**

- **Fair vergleichen**
- **Projekt umsetzen**

## Lernziele

Nach der Bearbeitung können Sie:

- ein ML-Projekt mit Nutzerfrage, Zielwert, Erfolgskriterium, Datenumfang und Grenzen klar planen.
- Modelle mit identischen Splits, Baselines, Metriken, Laufzeit- und Komplexitätsmessungen fair vergleichen.
- Teilgruppenleistung, Datenverschiebung, sichere Inferenz, Reproduzierbarkeit, Datenschutz und Modellgrenzen dokumentieren.

## Geprüfte Fähigkeiten

- Projektplanung, Datenvalidierung, EDA, Splitstrategie und Baseline
- fairer Modell- und Schwellenvergleich, Fehleranalyse, Laufzeit und Parameterzahl
- Teilgruppenmetriken, Driftprüfung, joblib-Persistenz, Eingabevalidierung und Modellkarte

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** Abschlussprojekt
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den Brustkrebs-Datensatz als lokale Projektdatenbasis und ergänzt eine rein technische Analysegruppe, die keine geschützte Personeneigenschaft darstellt. Training, Validierung und Test werden einmalig reproduzierbar festgelegt und danach von allen Modellen identisch verwendet.

In [ ]:
import json
import os
import platform
import tempfile
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

projekt_daten = load_breast_cancer(as_frame=True)
X_projekt = projekt_daten.data.copy()
y_projekt = projekt_daten.target.copy()
zielnamen = list(projekt_daten.target_names)

# Technische Analysegruppe für die Übung: kleiner bzw. großer gemessener Radius.
# Sie ist keine geschützte demografische Eigenschaft und ersetzt keine echte Fairnessanalyse.
radius_median = X_projekt["mean radius"].median()
technische_gruppe = pd.Series(
    np.where(X_projekt["mean radius"] <= radius_median, "Radius_klein", "Radius_gross"),
    index=X_projekt.index,
    name="Technische_Gruppe",
)

alle_indizes = np.arange(len(X_projekt))
idx_train, idx_test = train_test_split(
    alle_indizes,
    test_size=0.20,
    stratify=y_projekt,
    random_state=RANDOM_SEED,
)
idx_train, idx_val = train_test_split(
    idx_train,
    test_size=0.20,
    stratify=y_projekt.iloc[idx_train],
    random_state=RANDOM_SEED,
)

X_train = X_projekt.iloc[idx_train].copy()
y_train = y_projekt.iloc[idx_train].copy()
X_val = X_projekt.iloc[idx_val].copy()
y_val = y_projekt.iloc[idx_val].copy()
X_test = X_projekt.iloc[idx_test].copy()
y_test = y_projekt.iloc[idx_test].copy()
gruppe_test = technische_gruppe.iloc[idx_test].copy()

print("Train, Validierung, Test:", X_train.shape, X_val.shape, X_test.shape)
print("Klassen:", dict(enumerate(zielnamen)))

### Aufgabe 1: Projektauftrag, Nutzerfrage und Erfolgskriterien formulieren

Formulieren Sie einen kompakten Projektauftrag für diese Lehrdaten. Er muss enthalten:

1. die konkrete Vorhersagefrage,
2. die vorgesehene Nutzergruppe und mögliche Handlung nach einer Vorhersage,
3. die Beobachtungseinheit und den Zielwert,
4. mindestens zwei technische Erfolgskriterien,
5. mindestens drei Risiken oder Grenzen,
6. eine Begründung, weshalb das Modell keine autonome medizinische Diagnose treffen darf.

Verwenden Sie die Daten ausschließlich als Lehrbeispiel und vermeiden Sie klinische Leistungsversprechen.

> **Musterantwort und Interpretation**
>
> **Vorhersagefrage:** Kann ein Modell anhand der 30 bereits extrahierten Zellkernmerkmale zwischen den beiden Datensatzklassen unterscheiden?  
> **Nutzer und Handlung:** Lernende oder Datenanalysten nutzen das Modell zur Demonstration eines reproduzierbaren Klassifikationsworkflows. Eine Ausgabe darf höchstens eine technische Prüfpriorität auslösen, niemals eine Diagnose oder Behandlungsentscheidung.  
> **Beobachtungseinheit und Ziel:** Eine Zeile entspricht einer untersuchten Probe; der Zielwert ist die vom Datensatz bereitgestellte binäre Klasse.  
> **Technische Kriterien:** Das gewählte Modell soll die Dummy-Baseline deutlich übertreffen und auf dem unberührten Testdatensatz einen vorab definierten Recall sowie eine nachvollziehbare F1-Leistung erreichen. Laufzeit, Modellgröße und Teilgruppenwerte werden ebenfalls berichtet.  
> **Risiken und Grenzen:** Der Datensatz ist klein und historisch, enthält keine vollständige klinische Population, bildet reale Arbeitsabläufe nicht ab und besitzt keine geeigneten geschützten Gruppenattribute für eine medizinische Fairnessprüfung. Messprozess, Labelqualität und externe Übertragbarkeit sind nicht ausreichend belegt.  
> **Nicht-Verwendung:** Das Lehrmodell ist nicht klinisch validiert, nicht regulatorisch geprüft und berücksichtigt weder vollständige Patienteninformationen noch Folgen falsch positiver oder falsch negativer Entscheidungen. Es darf deshalb keine autonome medizinische Diagnose treffen.

### Aufgabe 2: Daten systematisch validieren und explorativ beschreiben

Erstellen Sie einen reproduzierbaren Datenqualitätsbericht für `X_projekt`, `y_projekt` und die technische Gruppe. Prüfen Sie mindestens:

- Form, Datentypen und Zielverteilung,
- Fehlwerte und Duplikate,
- nicht endliche Werte,
- Minima, Maxima und Quartile,
- Klassenverteilung innerhalb der technischen Gruppe,
- mögliche sehr starke lineare Korrelationen zwischen Merkmalen.

Visualisieren Sie die Zielverteilung und höchstens zehn stärkste absolute Merkmalskorrelationen. Formulieren Sie zwei konkrete Modellierungsfolgen aus den Befunden.

In [ ]:
# Erstellen Sie einen kompakten, aber systematischen Qualitätsbericht.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

qualitaetsbericht = {
    "Beobachtungen": len(X_projekt),
    "Merkmale": X_projekt.shape[1],
    "Fehlwerte": int(X_projekt.isna().sum().sum()),
    "Duplikate_X": int(X_projekt.duplicated().sum()),
    "Nicht_endliche_Werte": int((~np.isfinite(X_projekt.to_numpy())).sum()),
    "Klassenanzahl": y_projekt.value_counts().sort_index().to_dict(),
}
print(json.dumps(qualitaetsbericht, indent=2, ensure_ascii=False))

print("Datentypen:")
print(X_projekt.dtypes.value_counts())
print("Deskriptive Kennzahlen:")
display(X_projekt.describe().T[["min", "25%", "50%", "75%", "max"]].head(10))

klassen_nach_gruppe = pd.crosstab(
    technische_gruppe,
    y_projekt,
    normalize="index",
).rename(columns={0: zielnamen[0], 1: zielnamen[1]})
print("Relative Klassenverteilung nach technischer Gruppe:")
display(klassen_nach_gruppe.round(3))

korrelationen = X_projekt.corr().abs()
obere_maske = np.triu(np.ones_like(korrelationen, dtype=bool), k=1)
korrelationspaare = (
    korrelationen.where(obere_maske)
    .stack()
    .sort_values(ascending=False)
    .rename("Absolute_Korrelation")
    .reset_index()
    .rename(columns={"level_0": "Merkmal_A", "level_1": "Merkmal_B"})
)
print("Stärkste Korrelationen:")
display(korrelationspaare.head(10).round(3))

ziel_zaehlung = y_projekt.value_counts().sort_index()
plt.bar([zielnamen[i] for i in ziel_zaehlung.index], ziel_zaehlung.values)
plt.ylabel("Anzahl")
plt.title("Zielverteilung")
plt.show()

> **Musterantwort und Interpretation**
>
> Erstens unterscheiden sich Merkmalsgrößenordnungen stark, weshalb distanz- und regularisierungsabhängige lineare Modelle innerhalb einer Pipeline standardisiert werden sollten. Zweitens bestehen sehr hohe Korrelationen zwischen mehreren Größen- und Umfangsmerkmalen. Das kann Koeffizienten einzelner Merkmale instabil machen und verlangt vorsichtige Interpretation, auch wenn die Vorhersageleistung gut ist. Die Klassen sind nicht perfekt ausgeglichen, daher sollten neben Accuracy auch Recall, Precision, F1 und eine Baseline betrachtet werden.

### Aufgabe 3: Baseline und Modelle unter identischen Bedingungen vergleichen

Vergleichen Sie auf den identischen Splits:

- `DummyClassifier(strategy="most_frequent")`,
- skalierte logistische Regression,
- einen begrenzten Entscheidungsbaum,
- einen Random Forest mit höchstens 150 Bäumen.

Messen Sie Trainingszeit, Vorhersagezeit, Accuracy, Precision, Recall, F1 und ROC-AUC auf der Validierung. Berichten Sie außerdem eine nachvollziehbare Parameter- oder Komplexitätszahl. Speichern Sie alle angepassten Modelle und Validierungswahrscheinlichkeiten für spätere Aufgaben.

In [ ]:
def modellkomplexitaet(modell):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def modellkomplexitaet(modell):
    """Liefert eine grobe, modellspezifische Komplexitätszahl."""
    letzter_schritt = modell.steps[-1][1] if hasattr(modell, "steps") else modell
    if hasattr(letzter_schritt, "coef_"):
        return int(letzter_schritt.coef_.size + letzter_schritt.intercept_.size)
    if hasattr(letzter_schritt, "tree_"):
        return int(letzter_schritt.tree_.node_count)
    if hasattr(letzter_schritt, "estimators_"):
        return int(sum(baum.tree_.node_count for baum in letzter_schritt.estimators_))
    return np.nan

projekt_modelle = {
    "Mehrheitsbaseline": DummyClassifier(strategy="most_frequent"),
    "Logistische Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
    ),
    "Entscheidungsbaum": DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=8,
        random_state=RANDOM_SEED,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

angepasste_modelle = {}
val_wahrscheinlichkeiten = {}
vergleichszeilen = []
for name, modell in projekt_modelle.items():
    start = time.perf_counter()
    modell.fit(X_train, y_train)
    trainingszeit = time.perf_counter() - start

    start = time.perf_counter()
    if hasattr(modell, "predict_proba"):
        probs = modell.predict_proba(X_val)[:, 1]
    else:
        probs = modell.predict(X_val).astype(float)
    vorhersagezeit = time.perf_counter() - start
    pred = (probs >= 0.5).astype(int)

    angepasste_modelle[name] = modell
    val_wahrscheinlichkeiten[name] = probs
    # ROC-AUC benötigt mindestens zwei unterschiedliche Scorewerte.
    auc_wert = roc_auc_score(y_val, probs) if len(np.unique(probs)) > 1 else 0.5
    vergleichszeilen.append(
        {
            "Modell": name,
            "Trainingszeit_s": trainingszeit,
            "Vorhersagezeit_s": vorhersagezeit,
            "Accuracy": accuracy_score(y_val, pred),
            "Precision": precision_score(y_val, pred, zero_division=0),
            "Recall": recall_score(y_val, pred, zero_division=0),
            "F1": f1_score(y_val, pred, zero_division=0),
            "ROC_AUC": auc_wert,
            "Komplexitaetszahl": modellkomplexitaet(modell),
        }
    )

validierungsvergleich = pd.DataFrame(vergleichszeilen).sort_values("F1", ascending=False)
display(validierungsvergleich.round(5))

> **Musterantwort und Interpretation**
>
> Unterschiedliche Beobachtungen können einen Modellvergleich stärker beeinflussen als die Modellwahl selbst. Ein Modell könnte zufällig auf einem leichteren Testanteil bewertet werden. Identische Splits, Merkmale, Zieldefinitionen, Schwellenregeln und Metriken isolieren den Einfluss des Modellansatzes besser. Laufzeit- und Komplexitätsmessungen sollten ebenfalls unter möglichst gleichen Bedingungen erfolgen.

### Aufgabe 4: Schwelle ausschließlich mit Validierungsdaten wählen

Wählen Sie aus den probabilistischen Nicht-Baseline-Modellen das Modell mit dem höchsten Validierungs-F1. Suchen Sie auf der Validierung über Schwellen von 0.10 bis 0.90 eine Schwelle, die Precision von mindestens 0.85 erreicht und unter diesen Kandidaten maximalen Recall liefert.

Fixieren Sie anschließend Modell und Schwelle. Bewerten Sie genau diese Entscheidung einmalig auf dem Testdatensatz und vergleichen Sie sie mit der Standardschwelle 0.50. Erstellen Sie beide Konfusionsmatrizen.

In [ ]:
# Testdaten dürfen erst nach Abschluss der Modell- und Schwellenwahl verwendet werden.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

wahrscheinlichkeitsmodelle = validierungsvergleich[
    validierungsvergleich["Modell"] != "Mehrheitsbaseline"
].copy()
gewaehlter_modellname = wahrscheinlichkeitsmodelle.iloc[0]["Modell"]
gewaehltes_modell = angepasste_modelle[gewaehlter_modellname]
val_probs_gewaehlt = val_wahrscheinlichkeiten[gewaehlter_modellname]

schwellenzeilen = []
for schwelle in np.linspace(0.10, 0.90, 81):
    pred = (val_probs_gewaehlt >= schwelle).astype(int)
    schwellenzeilen.append(
        {
            "Schwelle": schwelle,
            "Precision": precision_score(y_val, pred, zero_division=0),
            "Recall": recall_score(y_val, pred, zero_division=0),
            "F1": f1_score(y_val, pred, zero_division=0),
        }
    )
schwellentabelle = pd.DataFrame(schwellenzeilen)
zulaessig = schwellentabelle[schwellentabelle["Precision"] >= 0.85]
if len(zulaessig) > 0:
    beste_schwellenzeile = zulaessig.sort_values(
        ["Recall", "F1", "Schwelle"],
        ascending=[False, False, True],
    ).iloc[0]
else:
    print("Keine Schwelle erreicht Precision 0.85; F1-Maximum wird als Fallback verwendet.")
    beste_schwellenzeile = schwellentabelle.loc[schwellentabelle["F1"].idxmax()]

gewaehlte_schwelle = float(beste_schwellenzeile["Schwelle"])
print("Gewähltes Modell:", gewaehlter_modellname)
print("Gewählte Schwelle:", round(gewaehlte_schwelle, 3))
display(beste_schwellenzeile.to_frame().T.round(3))

test_probs = gewaehltes_modell.predict_proba(X_test)[:, 1]
test_pred_standard = (test_probs >= 0.50).astype(int)
test_pred_gewaehlt = (test_probs >= gewaehlte_schwelle).astype(int)

def schwellenmetriken(name, pred):
    return {
        "Variante": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    }

test_schwellenvergleich = pd.DataFrame(
    [
        schwellenmetriken("Schwelle 0.50", test_pred_standard),
        schwellenmetriken(f"Validierungsschwelle {gewaehlte_schwelle:.2f}", test_pred_gewaehlt),
    ]
)
display(test_schwellenvergleich.round(3))

for titel, pred in [
    ("Standardschwelle 0.50", test_pred_standard),
    (f"Gewählte Schwelle {gewaehlte_schwelle:.2f}", test_pred_gewaehlt),
]:
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(titel)
    plt.show()

> **Musterantwort und Interpretation**
>
> Validierungs- und Testdaten sind endliche Stichproben. Modell und Schwelle können teilweise an zufällige Besonderheiten der Validierung angepasst sein, obwohl der Test unberührt blieb. Die Abweichung ist normale Schätzunsicherheit. Größere Datenmengen, wiederholte Validierung, Konfidenzintervalle und externe Tests liefern robustere Aussagen.

### Aufgabe 5: Teilgruppenleistung und Datenverschiebung prüfen

Bewerten Sie das fixierte Modell mit der gewählten Schwelle getrennt für `Radius_klein` und `Radius_gross`. Berichten Sie Anzahl, positive Zielrate, Accuracy, Precision, Recall, F1 und False-Positive-Rate.

Erzeugen Sie danach eine simulierte Driftkopie des Testdatensatzes, in der die ersten drei Merkmale jeweils um 0.75 Trainingsstandardabweichungen erhöht werden. Vergleichen Sie Merkmalsmittelwerte, mittlere vorhergesagte Wahrscheinlichkeit und positive Vorhersagerate vor und nach der Drift. Interpretieren Sie die technische Analyse vorsichtig und erklären Sie, warum sie keine demografische Fairnessprüfung ersetzt.

In [ ]:
def teilgruppenbericht(y_wahr, probs, gruppen, schwelle):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def teilgruppenbericht(y_wahr, probs, gruppen, schwelle):
    y_array = np.asarray(y_wahr)
    probs_array = np.asarray(probs)
    gruppen_array = np.asarray(gruppen)
    zeilen = []
    for gruppenname in np.unique(gruppen_array):
        maske = gruppen_array == gruppenname
        pred = (probs_array[maske] >= schwelle).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_array[maske], pred, labels=[0, 1]).ravel()
        zeilen.append(
            {
                "Gruppe": gruppenname,
                "Anzahl": int(maske.sum()),
                "Positive_Zielrate": float(y_array[maske].mean()),
                "Accuracy": accuracy_score(y_array[maske], pred),
                "Precision": precision_score(y_array[maske], pred, zero_division=0),
                "Recall": recall_score(y_array[maske], pred, zero_division=0),
                "F1": f1_score(y_array[maske], pred, zero_division=0),
                "False_Positive_Rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
            }
        )
    return pd.DataFrame(zeilen)

subgruppen = teilgruppenbericht(y_test, test_probs, gruppe_test, gewaehlte_schwelle)
display(subgruppen.round(3))

X_test_drift = X_test.copy()
verschobene_merkmale = list(X_train.columns[:3])
train_std = X_train[verschobene_merkmale].std()
X_test_drift[verschobene_merkmale] = X_test_drift[verschobene_merkmale] + 0.75 * train_std

drift_probs = gewaehltes_modell.predict_proba(X_test_drift)[:, 1]
drift_pred = (drift_probs >= gewaehlte_schwelle).astype(int)

mittelwertvergleich = pd.DataFrame(
    {
        "Training": X_train[verschobene_merkmale].mean(),
        "Test_original": X_test[verschobene_merkmale].mean(),
        "Test_drift": X_test_drift[verschobene_merkmale].mean(),
    }
)
print("Merkmalsmittelwerte:")
display(mittelwertvergleich.round(3))

drift_bericht = pd.DataFrame(
    {
        "Datensatz": ["Originaltest", "Simulierter Drift"],
        "Mittlere_Wahrscheinlichkeit": [test_probs.mean(), drift_probs.mean()],
        "Positive_Vorhersagerate": [test_pred_gewaehlt.mean(), drift_pred.mean()],
    }
)
display(drift_bericht.round(3))

> **Musterantwort und Interpretation**
>
> Die verwendete Gruppe ist eine technische Unterteilung nach einem Messmerkmal und keine geschützte Personengruppe. Außerdem können kleine Teilgruppen große statistische Unsicherheit besitzen. Umfassende Fairness erfordert geeignete Attribute, Beteiligung betroffener Gruppen, Prüfung unterschiedlicher Schadensarten, Datenentstehung, Zugänglichkeit, Schwellenfolgen und rechtlichen Kontext. Gleiche Kennzahlen allein beseitigen keine strukturellen Risiken.

### Aufgabe 6: Modell, Metadaten und sichere Inferenz reproduzierbar speichern

Speichern Sie das fixierte Modell mit `joblib` und eine JSON-Metadatendatei mit mindestens Bibliotheksversionen, Zufallsseed, Merkmalsnamen, Zielbedeutung, Schwelle, Trainingsgrößen und Validierungsentscheidung. Laden Sie beides neu und prüfen Sie identische Testwahrscheinlichkeiten.

Implementieren Sie `safe_predict_one`, die eine einzelne Zeile als Dictionary oder Series akzeptiert, fehlende oder zusätzliche Merkmale erkennt, Reihenfolge festlegt, nicht endliche Werte ablehnt und eine Warnung ausgibt, wenn ein Wert außerhalb des beobachteten Trainingsbereichs liegt. Geben Sie Wahrscheinlichkeit, Klasse, Schwelle und Warnungen zurück.

In [ ]:
def safe_predict_one(row, modell, metadaten, trainings_min, trainings_max):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def safe_predict_one(row, modell, metadaten, trainings_min, trainings_max):
    """Validiert Schema und Werte vor einer einzelnen Modellvorhersage."""
    series = pd.Series(row, dtype="float64")
    erwartete_merkmale = list(metadaten["merkmalsnamen"])
    fehlend = sorted(set(erwartete_merkmale) - set(series.index))
    zusaetzlich = sorted(set(series.index) - set(erwartete_merkmale))
    if fehlend:
        raise ValueError(f"Fehlende Merkmale: {fehlend}")
    if zusaetzlich:
        raise ValueError(f"Unerwartete Merkmale: {zusaetzlich}")

    geordnet = series[erwartete_merkmale].astype(float)
    if not np.isfinite(geordnet.to_numpy()).all():
        raise ValueError("Die Eingabe enthält NaN oder unendliche Werte.")

    warnungen_liste = []
    ausserhalb = (geordnet < trainings_min[erwartete_merkmale]) | (geordnet > trainings_max[erwartete_merkmale])
    if ausserhalb.any():
        warnungen_liste.append(
            "Mindestens ein Merkmal liegt außerhalb des im Training beobachteten Bereichs: "
            + ", ".join(geordnet.index[ausserhalb].tolist())
        )

    eingabe_df = geordnet.to_frame().T
    wahrscheinlichkeit = float(modell.predict_proba(eingabe_df)[0, 1])
    schwelle = float(metadaten["schwelle"])
    return {
        "wahrscheinlichkeit_positive_klasse": wahrscheinlichkeit,
        "vorhergesagte_klasse": int(wahrscheinlichkeit >= schwelle),
        "schwelle": schwelle,
        "warnungen": warnungen_liste,
    }

metadaten = {
    "projekt": "Lehrprojekt Brustkrebsdatensatz",
    "modellname": gewaehlter_modellname,
    "schwelle": gewaehlte_schwelle,
    "merkmalsnamen": list(X_train.columns),
    "zielnamen": zielnamen,
    "random_seed": RANDOM_SEED,
    "trainingsbeispiele": len(X_train),
    "validierungsbeispiele": len(X_val),
    "testbeispiele": len(X_test),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "auswahlregel": "Höchster Validierungs-F1, danach Recall-Maximum bei Precision mindestens 0.85",
}
trainings_min = X_train.min()
trainings_max = X_train.max()

with tempfile.TemporaryDirectory() as temp_ordner:
    modell_pfad = os.path.join(temp_ordner, "projektmodell.joblib")
    meta_pfad = os.path.join(temp_ordner, "modell_metadaten.json")
    joblib.dump(gewaehltes_modell, modell_pfad)
    with open(meta_pfad, "w", encoding="utf-8") as datei:
        json.dump(metadaten, datei, ensure_ascii=False, indent=2)

    geladenes_modell = joblib.load(modell_pfad)
    with open(meta_pfad, "r", encoding="utf-8") as datei:
        geladene_metadaten = json.load(datei)

    geladene_probs = geladenes_modell.predict_proba(X_test)[:, 1]
    print("Wahrscheinlichkeiten identisch:", np.allclose(test_probs, geladene_probs))
    assert np.allclose(test_probs, geladene_probs)

    sichere_ausgabe = safe_predict_one(
        X_test.iloc[0].to_dict(),
        geladenes_modell,
        geladene_metadaten,
        trainings_min,
        trainings_max,
    )
    print(json.dumps(sichere_ausgabe, indent=2, ensure_ascii=False))

> **Musterantwort und Interpretation**
>
> Solche Dateien können beim Laden Python-Objekte rekonstruieren und sind nicht als sicherer Datenaustausch mit unbekannten Quellen gedacht. Es dürfen nur Artefakte aus vertrauenswürdiger, kontrollierter Herkunft geladen werden. Prüfsummen, signierte Artefakte, Zugriffskontrollen, isolierte Umgebungen und dokumentierte Versionen reduzieren zusätzliche Risiken.

### Aufgabe 7: Integrationsaufgabe: gezielte Verbesserung und Modellkarte erstellen

Führen Sie genau ein begründetes Verbesserungsexperiment durch, ohne den Testdatensatz zur Auswahl zu verwenden. Vergleichen Sie für die logistische Regression `class_weight=None` und `class_weight="balanced"` auf der Validierung mit der bereits festgelegten Schwellenregel. Wählen Sie nur bei nachvollziehbarer Validierungsverbesserung eine neue Version und bewerten Sie diese danach auf dem Test.

Erstellen Sie anschließend programmgesteuert eine kompakte Modellkarte als Markdowntext. Sie muss Zweck, Nicht-Verwendungen, Daten, Split, Baseline, gewählte Version, Schwelle, Testmetriken, Teilgruppenbefunde, Driftbefund, Ressourcen, Datenschutz, bekannte Grenzen, Überwachungsplan und Reproduzierbarkeitsinformationen enthalten.

In [ ]:
def finde_schwelle_mit_precision(y_wahr, probs, minimum_precision=0.85):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def finde_schwelle_mit_precision(y_wahr, probs, minimum_precision=0.85):
    zeilen = []
    for schwelle in np.linspace(0.10, 0.90, 81):
        pred = (probs >= schwelle).astype(int)
        zeilen.append(
            {
                "schwelle": float(schwelle),
                "precision": precision_score(y_wahr, pred, zero_division=0),
                "recall": recall_score(y_wahr, pred, zero_division=0),
                "f1": f1_score(y_wahr, pred, zero_division=0),
            }
        )
    tabelle = pd.DataFrame(zeilen)
    zulaessig = tabelle[tabelle["precision"] >= minimum_precision]
    if len(zulaessig) == 0:
        return tabelle.loc[tabelle["f1"].idxmax()]
    return zulaessig.sort_values(
        ["recall", "f1", "schwelle"],
        ascending=[False, False, True],
    ).iloc[0]

verbesserungsvarianten = {}
verbesserungsberichte = []
for gewichtung in [None, "balanced"]:
    name = "LogReg Standard" if gewichtung is None else "LogReg class_weight=balanced"
    modell = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2500,
            class_weight=gewichtung,
            random_state=RANDOM_SEED,
        ),
    )
    modell.fit(X_train, y_train)
    val_probs = modell.predict_proba(X_val)[:, 1]
    schwellen_resultat = finde_schwelle_mit_precision(y_val, val_probs, minimum_precision=0.85)
    verbesserungsvarianten[name] = {
        "modell": modell,
        "schwelle": float(schwellen_resultat["schwelle"]),
    }
    verbesserungsberichte.append(
        {
            "Version": name,
            "Schwelle": schwellen_resultat["schwelle"],
            "Val_Precision": schwellen_resultat["precision"],
            "Val_Recall": schwellen_resultat["recall"],
            "Val_F1": schwellen_resultat["f1"],
        }
    )

verbesserungstabelle = pd.DataFrame(verbesserungsberichte).sort_values(
    ["Val_Recall", "Val_F1"], ascending=False
)
display(verbesserungstabelle.round(3))

beste_version = verbesserungstabelle.iloc[0]["Version"]
finales_modell = verbesserungsvarianten[beste_version]["modell"]
finale_schwelle = verbesserungsvarianten[beste_version]["schwelle"]
finale_probs = finales_modell.predict_proba(X_test)[:, 1]
finale_pred = (finale_probs >= finale_schwelle).astype(int)

finale_metriken = {
    "Accuracy": accuracy_score(y_test, finale_pred),
    "Precision": precision_score(y_test, finale_pred, zero_division=0),
    "Recall": recall_score(y_test, finale_pred, zero_division=0),
    "F1": f1_score(y_test, finale_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, finale_probs),
}
print("Finale Version:", beste_version)
print("Finale Schwelle:", round(finale_schwelle, 3))
print("Finale Testmetriken:", {k: round(v, 3) for k, v in finale_metriken.items()})

# Die Modellkarte wird aus den tatsächlich berechneten Ergebnissen erzeugt.
modellkarte = f"""
# Modellkarte: Lehrprojekt Zellkernmerkmale

## Zweck
Demonstration eines reproduzierbaren binären Klassifikationsworkflows auf einem öffentlichen Lehrdatensatz.

## Nicht vorgesehene Verwendungen
Keine klinische Diagnose, keine Therapieentscheidung, keine autonome Priorisierung realer Patientinnen oder Patienten und keine Übertragung auf andere Einrichtungen ohne externe Validierung.

## Daten und Split
- {len(X_projekt)} Beobachtungen mit {X_projekt.shape[1]} numerischen Merkmalen
- Training: {len(X_train)}, Validierung: {len(X_val)}, Test: {len(X_test)}
- Reproduzierbarer stratifizierter Split mit Seed {RANDOM_SEED}
- Vorverarbeitung wird innerhalb der Pipeline nur auf Training angepasst

## Baseline und gewählte Version
- Baseline: häufigste Klasse
- Finale Version: {beste_version}
- Entscheidungsschwelle: {finale_schwelle:.3f}
- Auswahl erfolgte ausschließlich mit Validierungsdaten

## Testleistung
- Accuracy: {finale_metriken['Accuracy']:.3f}
- Precision: {finale_metriken['Precision']:.3f}
- Recall: {finale_metriken['Recall']:.3f}
- F1: {finale_metriken['F1']:.3f}
- ROC-AUC: {finale_metriken['ROC_AUC']:.3f}

## Teilgruppen- und Driftprüfung
Die technische Radiusgruppe zeigte unterschiedliche Zielraten und teils unterschiedliche Fehlerkennzahlen. Sie ist keine demografische Fairnessanalyse. Eine simulierte Verschiebung ausgewählter Merkmale veränderte Wahrscheinlichkeiten und Vorhersageraten und zeigt den Bedarf an Eingangs- und Leistungsüberwachung.

## Ressourcen
Kleine scikit-learn-Pipeline für CPU-Inferenz. Trainings- und Vorhersagezeiten sowie eine modellspezifische Komplexitätszahl wurden im fairen Vergleich dokumentiert.

## Datenschutz und Sicherheit
Der Lehrdatensatz enthält keine hier verwendeten direkten Identifikatoren. Reale Anwendungen müssten Datenminimierung, Zugriffsrechte, Aufbewahrung, Protokollierung und sichere Artefaktherkunft regeln. Serialisierte Modelle dürfen nur aus vertrauenswürdigen Quellen geladen werden.

## Bekannte Grenzen
Kleine historische Stichprobe, fehlende externe Validierung, unvollständige klinische Informationen, mögliche Korrelationen und Verteilungsverschiebungen, keine geeigneten geschützten Attribute sowie statistische Unsicherheit der Teilgruppenwerte.

## Überwachung
Eingabeschema, Fehlwerte, Wertebereiche, Merkmalsdrift, Vorhersagerate, Recall und Fehlalarme regelmäßig prüfen. Bei deutlicher Drift oder Leistungsabfall muss das Modell gestoppt, fachlich untersucht und gegebenenfalls neu validiert werden.

## Reproduzierbarkeit
Python {platform.python_version()}, scikit-learn {sklearn.__version__}, NumPy {np.__version__}, Seed {RANDOM_SEED}. Merkmalsnamen, Schwelle, Splitgrößen und Auswahlregel werden zusammen mit dem Modell versioniert.
""".strip()

print(modellkarte)

> **Musterantwort und Interpretation**
>
> Vor jedem realen Einsatz wäre eine unabhängige externe Validierung auf Daten aus dem vorgesehenen Arbeitsablauf erforderlich. Dabei müssten Beobachtungseinheit, Messprozess, Zieldefinition, Fehlerfolgen, repräsentative Teilgruppen, Datenschutz und menschliche Entscheidungsverantwortung gemeinsam geprüft werden. Erst danach wäre eine prospektive, überwachte Pilotphase mit klaren Stopkriterien vertretbar. Die gute Leistung eines kleinen Lehrdatensatzes reicht dafür nicht aus.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?